In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

In [5]:
# 1. Cargar el dataset con tus 3.444 compuestos que superaron el umbral de 6.0
df_candidatos = pd.read_csv('../results/TOP_inhibidores_FabI_CMNPD.csv')

In [ ]:
# 2. Definir la función para evaluar la Regla de Lipinski
def lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False 
        
    # Calcular los 4 parámetros exactos usando el módulo Descriptors de RDKit
    peso_molecular = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    donadores_h = Descriptors.NumHDonors(mol)
    aceptores_h = Descriptors.NumHAcceptors(mol)

    # 3. Aplicar filtro estricto (0 violaciones permitidas)
    if (peso_molecular < 500) and (logp < 5) and (donadores_h <= 5) and (aceptores_h <= 10):
        return True
    else:
        return False

print("Calculando parámetros de Lipinski para todos los compuestos...")

In [13]:
# 4. Aplicar la función a la columna SMILES
df_candidatos['Reglas_Lipinski'] = df_candidatos['Smiles'].apply(lipinski)

In [ ]:
# 5. Filtrar el DataFrame quedándonos solo con los que cumplen la regla
df_lipinski = df_candidatos[df_candidatos['Reglas_Lipinski'] == True].copy()
df_lipinski = df_lipinski.drop(columns=['Reglas_Lipinski']).reset_index(drop=True)

,Smiles,pChEMBL_Fisico,pChEMBL_MACCS,Media_Predicciones


In [9]:
# 6. Guardar los compuestos viables como medicamentos orales
df_lipinski.to_csv('../results/CMNPD_Lipinski_Aprobados.csv', index=False)

print(f"\n¡Filtrado completado! De {len(df_candidatos)} candidatos iniciales, {len(df_lipinski)} cumplen estrictamente la Regla de Lipinski.")
df_lipinski.head()


¡Filtrado completado! De 3444 candidatos iniciales, 0 cumplen estrictamente la Regla de Lipinski.


,Smiles,pChEMBL_Fisico,pChEMBL_MACCS,Media_Predicciones


In [14]:
# 2. Definir una función de Lipinski más realista (contando violaciones)
def calcular_violaciones_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 99 # Valor alto para descartar si el SMILES es inválido
        
    violaciones = 0
    
    if Descriptors.MolWt(mol) > 500:
        violaciones += 1
    if Descriptors.MolLogP(mol) > 5:
        violaciones += 1
    if Descriptors.NumHDonors(mol) > 5:
        violaciones += 1
    if Descriptors.NumHAcceptors(mol) > 10:
        violaciones += 1
        
    return violaciones

print("Calculando el número de violaciones de Lipinski por compuesto...")

Calculando el número de violaciones de Lipinski por compuesto...


In [15]:
# 3. Aplicar la función
df_candidatos['Violaciones_Lipinski'] = df_candidatos['Smiles'].apply(calcular_violaciones_lipinski)

In [16]:
# 4. APLICAR FILTRO FLEXIBLE: Nos quedamos con los que tienen 1 violación o menos (puedes poner 2 si siguen siendo pocos)
max_violaciones_permitidas = 1
df_lipinski = df_candidatos[df_candidatos['Violaciones_Lipinski'] <= max_violaciones_permitidas].copy()

In [17]:
# Opcional: limpiar la columna y reiniciar el índice
df_lipinski = df_lipinski.drop(columns=['Violaciones_Lipinski']).reset_index(drop=True)


In [19]:
# 5. Guardar los compuestos
df_lipinski.to_csv('../results/CMNPD_Lipinski_Flexible.csv', index=False)

print(f"\n¡Filtrado completado! De {len(df_candidatos)} candidatos iniciales, {len(df_lipinski)} cumplen la regla permitiendo hasta {max_violaciones_permitidas} violación(es).")


¡Filtrado completado! De 3444 candidatos iniciales, 1239 cumplen la regla permitiendo hasta 1 violación(es).


Selección de Scaffolds